[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Optimization/Convex_Optimization_2.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Convex Optimization II: Duality, Proximal Methods & ADMM

The sequel [Optimization](./Optimization.ipynb) earned: duality (every convex problem has a shadow twin whose gap certifies optimality), proximal operators (the theory [ISTA](../../Intro_DSP/Compressed_Sensing.ipynb) was waiting for), and ADMM — the splitting method behind large-scale and distributed solvers.

## 1. Pre-requisites

[Optimization](./Optimization.ipynb) S1–S3 (convexity, GD, Lagrange). [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb) motivates everything here.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Duality & the Certificate* (~40 min)
**Goal:** build the dual problem; use the duality gap as a PROOF of optimality.
**Builds on:** [Optimization](./Optimization.ipynb) S3. &nbsp; **Feeds into:** Session 2 (proximal).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Duality & the Certificate</b></summary>

**Timing (~40 min).** 10 min weak duality · 10 min Slater and strong duality · 12 min the demo and its certificate · 8 min complementary slackness.

**Board first — weak duality is nearly free, so prove it.** For any $\lambda \ge 0$ and any feasible $x$, $\mathcal{L}(x,\lambda) \le f(x)$ because the constraint terms are non-positive; take the infimum over $x$ and $g(\lambda) \le f(x)$ for every feasible $x$. Three lines, no convexity assumed, universally true. Emphasise that: **weak duality holds for non-convex problems too**, which is why dual bounds are used in integer programming and combinatorial optimisation where nothing else is available.

**Then strong duality as the convex bonus.** For a convex problem with a strictly feasible point (Slater's condition), the best lower bound actually *touches* the optimum — the gap is zero. Convexity is what buys the tightness, not the bound itself.

**The certificate is the practical superpower — frame it as such.** Any dual-feasible $\lambda$ produces a *provable* lower bound. So if your primal and dual values agree to $\varepsilon$, you have **proven** your answer is $\varepsilon$-optimal, with no faith in the solver required. Ask the room how else you would know an iterative solver had converged: you would watch the iterates stop moving, which proves nothing. A duality gap is a mathematical guarantee, and it is why serious solvers report one.

**Handle the negative gap honestly — a student will notice.** The printed gap is $-1.11\times10^{-15}$, and weak duality says the gap must be $\ge 0$. A negative value would be a contradiction if it were real; at $10^{-15}$ it is floating-point noise on a quantity of size 1.4. Say so plainly. It is a good moment to teach that agreement *at machine precision* can present as a small negative number, and that the right response is to check the magnitude against $\varepsilon_{\text{machine}}$ rather than to panic.

**Complementary slackness deserves its own minute.** $\max|\lambda_i \cdot \text{slack}_i| = 6.3\times10^{-16}$ says that for every constraint, either the multiplier is zero or the constraint is tight. Read it as economics: $\lambda_i$ is the *shadow price* of constraint $i$, and a constraint that is not binding has price zero. That interpretation makes duals feel like meaningful objects rather than algebraic bookkeeping, and it is how practitioners actually use them — a large $\lambda_i$ tells you which constraint to relax first.

**Ask the room.** "The dual was solved by projected gradient ascent on $\lambda \ge 0$. Why is that easy?" Because the dual function is *always concave*, regardless of the primal's structure — so the dual is a concave maximisation even when the primal is nasty. That is the other reason duality matters.
</details>

## 2. Every Problem's Shadow

💡 **Intuition.** The Lagrangian dual $g(\lambda) = \inf_x \mathcal{L}(x, \lambda)$ is a lower bound on the optimum for *every* $\lambda \ge 0$ (**weak duality** — trivial to prove, universally true). For convex problems with a strictly feasible point (Slater), the best lower bound *touches*: **strong duality**, gap zero. The practical superpower: any dual-feasible λ gives a *certificate* — if primal and dual values are within ε, you have PROVEN your answer is ε-optimal, no faith in the solver required.

In [2]:
# Certified optimality on a QP:  min ½xᵀPx + qᵀx  s.t.  Ax ≤ b
n_v, m_c = 20, 12
M = rng.standard_normal((n_v, n_v)); P = M @ M.T + np.eye(n_v)
q = rng.standard_normal(n_v)
A = rng.standard_normal((m_c, n_v)); b = rng.random(m_c) * 2

# dual of a QP: g(λ) = −½(q + Aᵀλ)ᵀ P⁻¹ (q + Aᵀλ) − bᵀλ,  λ ≥ 0 — maximize by projected GD
lamb = np.zeros(m_c)
Pinv = np.linalg.inv(P)
for _ in range(4000):
    x_of_lam = -Pinv @ (q + A.T @ lamb)
    grad = A @ x_of_lam - b                       # ∂g/∂λ
    lamb = np.maximum(0, lamb + 0.01 * grad)

x_dual = -Pinv @ (q + A.T @ lamb)
x_feas = x_dual.copy()                             # project tiny violations
viol = A @ x_feas - b
primal = 0.5 * x_feas @ P @ x_feas + q @ x_feas
dual   = -0.5 * (q + A.T@lamb) @ Pinv @ (q + A.T@lamb) - b @ lamb
print(f"primal value {primal:.6f}   dual value {dual:.6f}")
print(f"duality gap  {primal - dual:.2e}  → the answer is CERTIFIED {primal-dual:.0e}-optimal")
print(f"max constraint violation {viol.max():.2e};  complementary slackness "
      f"max|λᵢ·slackᵢ| = {np.abs(lamb * viol).max():.2e}")

primal value -1.421796   dual value -1.421796
duality gap  -1.11e-15  → the answer is CERTIFIED -1e-15-optimal
max constraint violation 2.22e-15;  complementary slackness max|λᵢ·slackᵢ| = 6.28e-16


**What just happened.** Primal and dual values agree to **fifteen decimal places** (both $-1.421796$), with a duality gap of $-1.11\times10^{-15}$, constraint violation $2.2\times10^{-15}$, and complementary slackness holding at $6.3\times10^{-16}$.

**The gap is the point, and it is a different kind of result from "the solver converged."** Weak duality guarantees that *any* dual-feasible $\lambda$ gives a lower bound on the optimum. So when the primal objective and the dual objective agree to $\varepsilon$, you have **proven** your answer is within $\varepsilon$ of optimal. Not observed, not plausibly inferred from iterates that stopped moving — proven, by an inequality that holds unconditionally.

That is why serious solvers report a duality gap and why it belongs in any optimisation pipeline you trust. An iterative method that has stopped changing might be at an optimum or might be stuck; the gap distinguishes those cases.

**Now the negative sign, because it should bother you.** Weak duality says primal $\ge$ dual, so the gap must be $\ge 0$ — and $-1.11\times10^{-15}$ is negative. If that number were real it would contradict a theorem. It is not real: it is floating-point noise on a quantity of magnitude 1.4, roughly $\varepsilon_{\text{machine}}$ relative. **Agreement at machine precision routinely presents as a small negative gap**, and the correct response is to compare the magnitude against $\varepsilon_{\text{machine}} \approx 2.2\times10^{-16}$ rather than to conclude the mathematics broke.

**Weak duality is cheaper than you might expect, and that matters.** The proof needs no convexity at all: for $\lambda \ge 0$ and feasible $x$, the constraint terms in $\mathcal{L}(x,\lambda)$ are non-positive, so $\mathcal{L}(x,\lambda) \le f(x)$, and taking the infimum gives $g(\lambda) \le f(x)$. Three lines. Consequently **dual bounds are available even for non-convex and integer problems**, where they are often the only rigorous lower bound anyone has — the basis of branch-and-bound.

What convexity plus Slater's condition adds is **tightness**: the best lower bound actually touches, so the gap closes to zero. Convexity buys the equality, not the bound.

**And complementary slackness has an economic reading worth carrying.** $\max|\lambda_i\,\text{slack}_i| \approx 0$ means that for each constraint, either its multiplier is zero or the constraint is tight. The multiplier $\lambda_i$ is the **shadow price** of constraint $i$ — how much the optimum would improve if you relaxed it by one unit — and a non-binding constraint is worth nothing. That makes the dual variables directly actionable: a large $\lambda_i$ tells you which constraint is costing you most, which is exactly how duals get used in resource allocation.

One structural note: the dual was maximised by *projected* gradient ascent, and that worked easily because **the dual function is always concave**, whatever the primal looks like. Concave maximisation over $\lambda \ge 0$ is a well-behaved problem by construction.

---
### 🕐 Session 2 of 3 — *Proximal Operators* (~40 min)
**Goal:** minimize smooth + nonsmooth: the prox map, soft-thresholding derived, ISTA justified.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (ADMM).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Proximal Operators</b></summary>

**Timing (~40 min).** 8 min why gradients fail on nonsmooth terms · 12 min the prox map and deriving soft-threshold · 10 min the verification · 10 min ISTA vs FISTA.

**Board first — state the problem gradients cannot solve.** $\|x\|_1$ has no derivative at zero, and zero is exactly where the interesting solutions live (that is the whole point of sparsity). So gradient descent is not merely inefficient here, it is undefined at the optimum. Ask the room what to do; subgradients are the usual first answer, and they work badly — convergence degrades to $O(1/\sqrt{k})$ and iterates never become exactly sparse.

**Then the prox, read aloud as a compromise.** $\mathrm{prox}_{\tau g}(v) = \arg\min_x\, g(x) + \frac{1}{2\tau}\|x-v\|^2$: *move toward $v$, but pay $g$*. Proximal gradient descent then alternates a gradient step on the smooth part with a prox step on the nonsmooth part — handling each piece with the tool that suits it.

**Derive the soft-threshold rather than quoting it; it is five minutes and it demystifies ISTA.** With $g = \lambda\|\cdot\|_1$ the objective separates per coordinate, so you solve a one-dimensional problem three times: $x > 0$, $x < 0$, $x = 0$. The answer is $\mathrm{sign}(v)\max(|v|-\tau, 0)$. Then the reveal — **that is exactly the line inside ISTA** from [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb). Students who met ISTA as a recipe now see it was proximal gradient descent all along, with the full convergence theory of [Optimization S2](./Optimization.ipynb) behind it.

**Praise the verification method.** The demo does not check the formula against another formula; it brute-forces the argmin over a fine grid and compares. Agreement at $6.7\times10^{-16}$ means the closed form is right, verified independently of the derivation. That is a genuinely different kind of check from algebraic re-derivation, and worth naming as a habit.

**On the ISTA/FISTA plot — insist on reading slopes, not heights.** The claim is $O(1/k)$ against $O(1/k^2)$, which on log-log axes is a difference in *slope*: FISTA's line should be about twice as steep. Have the room measure both slopes off the figure rather than observing that one curve is lower. A method that were merely lower by a constant would be a different and much weaker claim.

**And be precise about what momentum does and does not cost.** FISTA adds two lines and no extra gradient evaluations — the per-iteration cost is essentially identical. The acceleration is free in flops. Note also that FISTA is *not monotone*: its objective can increase on individual iterations, which alarms people watching a loss curve. That is expected behaviour for an accelerated method, not a bug.

**Connect it to the wider pattern if time allows.** The $O(1/k) \to O(1/k^2)$ jump from momentum is the same phenomenon as CG's $\kappa \to \sqrt\kappa$ in [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) and Nesterov momentum in [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb). Momentum extracting a square root is a recurring structural fact, not three coincidences.
</details>

## 3. Gradient Steps for the Non-Differentiable

💡 **Intuition.** For $f + g$ with $g$ nonsmooth (an L1 norm, a constraint indicator), define the **prox**: $\mathrm{prox}_{\tau g}(v) = \arg\min_x g(x) + \frac{1}{2\tau}\|x - v\|^2$ — 'move toward $v$, but pay $g$'. Proximal gradient descent alternates a gradient step on $f$ with a prox step on $g$; for $g = \lambda\|\cdot\|_1$ the prox is exactly **soft-thresholding** (derive it: the problem separates per coordinate, three cases, done) — so [ISTA](../../Intro_DSP/Compressed_Sensing.ipynb) was proximal gradient all along, with the full convergence theory of [Optimization S2](./Optimization.ipynb) behind it. FISTA adds momentum: $O(1/k) \to O(1/k^2)$.

In [3]:
# ORACLE: prox of L1 computed by brute-force minimization == soft threshold formula
tau_l = 0.7
v_grid = np.linspace(-3, 3, 61)
xs = np.linspace(-5, 5, 40001)
brute = [xs[np.argmin(tau_l*np.abs(xs) + 0.5*(xs - v)**2)] for v in v_grid]
formula = np.sign(v_grid) * np.maximum(np.abs(v_grid) - tau_l, 0)
print("max |brute-force prox − soft-threshold formula| =", np.abs(np.array(brute) - formula).max())
plt.figure(figsize=(6.5, 2.6))
plt.plot(v_grid, brute, "o", markersize=3, label="brute-force argmin")
plt.plot(v_grid, formula, "-", label="soft threshold")
plt.legend(); plt.title("the prox of λ|·|₁, derived and verified")
plt.tight_layout(); plt.show()

max |brute-force prox − soft-threshold formula| = 6.661338147750939e-16


/tmp/ipykernel_2980652/2397073675.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The closed-form soft-threshold matches a **brute-force minimisation** to $6.7\times10^{-16}$ — machine precision — across the whole range of inputs.

**Notice what kind of check that is.** The cell does not verify the formula against another formula, or re-derive the algebra a second way. It computes $\arg\min_x \tau|x| + \frac{1}{2}(x-v)^2$ by *searching a fine grid*, using no theory at all, and compares. Two genuinely independent routes to the same numbers. That is a much stronger form of verification than algebraic re-derivation, which can repeat the same mistake twice, and it is a habit worth copying.

**Why the prox exists at all.** $\|x\|_1$ has no derivative at zero — and zero is precisely where the interesting solutions live, since sparsity *means* coordinates being exactly zero. So gradient descent is not merely inefficient on this objective, it is undefined at the optimum. Subgradient methods work but poorly: convergence degrades to $O(1/\sqrt{k})$ and the iterates never become exactly sparse, only small.

The prox sidesteps this by solving a small proximal subproblem exactly rather than linearising:
$$\mathrm{prox}_{\tau g}(v) = \arg\min_x\, g(x) + \tfrac{1}{2\tau}\|x-v\|^2,$$
read as *move toward $v$, but pay $g$*. Proximal gradient descent alternates a gradient step on the smooth part with a prox step on the nonsmooth part, handling each with the appropriate tool.

**And for the L1 norm the prox is derivable in five minutes.** The objective separates per coordinate, so it is a one-dimensional problem solved three times ($x>0$, $x<0$, $x=0$), giving $\mathrm{sign}(v)\max(|v|-\tau, 0)$. Look at the plot: a flat region on $[-\tau, \tau]$ where the output is *exactly zero*, and unit-slope lines outside it, shifted toward the origin. That flat region is where sparsity comes from — it is not a small value, it is an exact zero — and it is the L1 ball's corner from [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb) appearing as an operation.

**The payoff sentence: ISTA was proximal gradient descent all along.** The line `c = np.sign(c) * np.maximum(np.abs(c) - lam*eta, 0)` in that workshop is exactly this prox, and the gradient step preceding it is exactly the smooth part. What was presented there as a working recipe is here revealed as an instance of a general method — which means the entire convergence theory of [Optimization S2](./Optimization.ipynb) applies to it, including the $O(1/k)$ rate and the FISTA acceleration in the next cell.

In [4]:
# ISTA vs FISTA on a LASSO problem — the promised O(1/k) vs O(1/k²)
m, n_f = 120, 400
A = rng.standard_normal((m, n_f)) / np.sqrt(m)
x_true = np.zeros(n_f); x_true[rng.choice(n_f, 10, replace=False)] = rng.standard_normal(10) * 2
b = A @ x_true + 0.01*rng.standard_normal(m)
lam_reg = 0.02
L_lip = np.linalg.norm(A, 2)**2

def obj(x): return 0.5*np.sum((A@x - b)**2) + lam_reg*np.abs(x).sum()
def soft(v, t): return np.sign(v)*np.maximum(np.abs(v)-t, 0)

hist = {}
x = np.zeros(n_f); hist["ISTA"] = []
for k in range(400):
    x = soft(x - (1/L_lip)*A.T@(A@x - b), lam_reg/L_lip)
    hist["ISTA"].append(obj(x))
x = np.zeros(n_f); y = x.copy(); t = 1.0; hist["FISTA"] = []
for k in range(400):
    x_new = soft(y - (1/L_lip)*A.T@(A@y - b), lam_reg/L_lip)
    t_new = (1 + np.sqrt(1 + 4*t*t))/2
    y = x_new + ((t-1)/t_new)*(x_new - x)
    x, t = x_new, t_new
    hist["FISTA"].append(obj(x))

f_star = min(min(v) for v in hist.values())
plt.figure(figsize=(7.5, 2.8))
for name, v in hist.items():
    plt.loglog(np.array(v) - f_star + 1e-12, label=name)
plt.legend(); plt.xlabel("iteration"); plt.ylabel("objective gap")
plt.title("momentum on a nonsmooth problem: FISTA's provable 1/k² vs ISTA's 1/k")
plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()

/tmp/ipykernel_2980652/540085333.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()


**What just happened.** Two solvers on the same LASSO problem, and on log-log axes the claim is about **slope**, not height: ISTA converges at $O(1/k)$, FISTA at $O(1/k^2)$, so FISTA's line should be roughly twice as steep.

Read it that way rather than "the orange curve is lower." A method that were merely lower by a constant factor would be a much weaker result — the gap between the two would stay fixed. A difference in *rate* means the gap widens without limit, so FISTA's advantage grows the longer you run.

**And momentum is free in flops.** Compare the two loops: FISTA adds a $t$ recursion and one extrapolation line, with **no extra gradient evaluations**. The per-iteration cost is essentially identical, so the acceleration costs two lines of code and nothing else. That is unusual — most speedups trade something — and it is why FISTA is the default for this problem class.

**One behaviour that alarms people, so it is worth predicting.** FISTA is **not monotone**: its objective can *increase* on individual iterations. Watch the orange curve closely and it wobbles rather than descending cleanly. That is expected for an accelerated method — the momentum term extrapolates past the current iterate and sometimes overshoots — and it is not a bug. Anyone watching a loss curve and expecting monotone descent will misdiagnose it.

**Where the acceleration comes from, conceptually.** Plain proximal gradient uses only the current point. FISTA extrapolates along the direction of recent progress, so it builds up velocity in consistently-downhill directions and damps oscillation across narrow valleys. That is the same mechanism as heavy-ball and Nesterov momentum, and it is the same structural fact as CG's $\kappa \to \sqrt\kappa$ in [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) and momentum's role in [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb). **Momentum extracting a square root is a recurring theorem, not three coincidences.**

**Note the step size is principled, not tuned.** Both loops use $1/L$ where $L = \|A\|_2^2$ is the Lipschitz constant of the smooth part's gradient — computable in advance, with a convergence guarantee attached. Compare with deep learning, where the learning rate is searched for empirically. Convex optimisation can tell you the right step size because the theory is complete enough to; that is a real advantage of working in this setting, and worth naming.

**And the $O(1/k^2)$ is optimal.** Nesterov proved that no first-order method — nothing using only gradients — can do better on this problem class in the worst case. FISTA is not just faster than ISTA; it attains the ceiling for its class of methods.

---
### 🕐 Session 3 of 3 — *ADMM: Split & Coordinate* (~40 min)
**Goal:** consensus optimization: solve pieces separately, agree via duals — the distributed workhorse.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: ADMM — Split & Coordinate</b></summary>

**Timing (~40 min).** 8 min the splitting idea · 12 min the three updates · 10 min the demo against the oracle · 10 min why this distributes.

**Board first — the reframe that makes ADMM obvious.** Write $\min f(x) + g(x)$ and ask what makes it hard: the two terms want different things and share a variable. Now *duplicate* the variable — $\min f(x) + g(z)$ subject to $x = z$ — and the terms are decoupled, at the price of a constraint. ADMM handles that constraint with a dual variable. The whole method is "split the variable, then negotiate."

**Walk the three updates by naming what each one is.** The $x$-update touches only $f$ (here a cached linear solve). The $z$-update touches only $g$ (here a prox — Session 2's soft-threshold, reused). The $u$-update is a running **disagreement ledger**: it accumulates $x - z$, and the longer the two halves disagree, the harder the dual pushes them together. Framing $u$ as a tally of unpaid disagreement makes the algorithm intuitive rather than mysterious.

**The key structural point: each subproblem stays easy even when the sum is nasty.** LASSO couples a least-squares term with an L1 term, and neither is hard alone. ADMM never solves them together. Ask the room what the $x$-update costs — one linear solve with a *cached* factorisation, so after the first iteration it is a triangular solve. That caching is why ADMM scales.

**Set the demo up as an oracle test, not a demonstration.** ADMM and FISTA are completely different algorithms — different updates, different theory, different convergence rates — and the check is whether they find the *same minimiser*. Agreement at $1.07\times10^{-14}$ says they do. The support recovery is the stronger claim: all ten indices exactly right, which is a combinatorial statement rather than a numerical one.

**Be honest about ADMM's convergence behaviour.** It converges reliably to modest accuracy and then *slowly*; it is not competitive with FISTA if you need many digits on a problem that fits in memory. Note that the demo runs ADMM for 300 iterations and FISTA for 4000, which is a fair reflection of that: ADMM gets close fast and FISTA is used here as the high-accuracy reference. Do not let students conclude ADMM is uniformly better.

**Then why anyone uses it: it distributes.** The $x$-update decomposes across data blocks that live on different machines; each solves its own piece and only the consensus variable $z$ and dual $u$ are exchanged. That is *consensus ADMM*, and it is the mathematical ancestor of the all-reduce pattern in [Distributed Training](../../Intro_GPU/Distributed_Training_2.ipynb). Ask what gets communicated — one vector per round, not the data — which is exactly why the pattern survives at datacenter scale.

**Mention the tuning wart.** $\rho$ must be chosen, and convergence speed is sensitive to it while the *answer* is not. Adaptive-$\rho$ schemes exist. It is the one genuinely fiddly part of an otherwise clean method.
</details>

## 4. Divide, Solve, Reconcile

💡 **Intuition.** ADMM attacks $\min f(x) + g(z)$ s.t. $x = z$ by alternating: minimize an *augmented* Lagrangian in $x$ (only $f$), then in $z$ (only $g$ — often a prox!), then nudge the dual $u \mathrel{+}= x - z$ — a running tally of disagreement that prices consensus. Each subproblem stays simple even when the sum is nasty, and the same template distributes across machines ([Distributed Training](../../Intro_GPU/Distributed_Training_2.ipynb)'s mathematical ancestor).

In [5]:
# LASSO by ADMM — must agree with FISTA's answer (our oracle)
rho = 1.0
x = np.zeros(n_f); z = np.zeros(n_f); u = np.zeros(n_f)
Q = np.linalg.inv(A.T @ A + rho*np.eye(n_f))          # cached factorization
for k in range(300):
    x = Q @ (A.T @ b + rho*(z - u))                   # smooth piece: a linear solve
    z = soft(x + u, lam_reg/rho)                      # nonsmooth piece: a prox
    u = u + x - z                                     # disagreement ledger
x_admm = z

x_fista = None
x = np.zeros(n_f); y = x.copy(); t = 1.0
for k in range(4000):
    x_new = soft(y - (1/L_lip)*A.T@(A@y - b), lam_reg/L_lip)
    t_new = (1 + np.sqrt(1 + 4*t*t))/2
    y = x_new + ((t-1)/t_new)*(x_new - x)
    x, t = x_new, t_new
x_fista = x

print(f"‖x_ADMM − x_FISTA‖∞ = {np.abs(x_admm - x_fista).max():.2e}   (same minimizer, two routes)")
print(f"support recovered: {[int(i) for i in np.where(np.abs(x_admm) > 0.05)[0]]}")
print(f"true support:      {[int(i) for i in np.where(np.abs(x_true) > 0)[0]]}")

‖x_ADMM − x_FISTA‖∞ = 1.07e-14   (same minimizer, two routes)
support recovered: [0, 1, 10, 54, 92, 113, 275, 356, 394, 398]
true support:      [0, 1, 10, 54, 92, 113, 275, 356, 394, 398]


**What just happened.** Two completely different algorithms, one answer. ADMM and FISTA agree to $1.07\times10^{-14}$, and the recovered support `[0, 1, 10, 54, 92, 113, 275, 356, 394, 398]` matches the planted truth **exactly**.

**The support match is the stronger claim.** Agreeing on ten real numbers to 14 decimals says the two solvers found the same point. Agreeing on *which ten of four hundred* coordinates are nonzero is a combinatorial statement — there are $\binom{400}{10} \approx 2\times10^{19}$ possible supports — and both methods, from different update rules and different theory, landed on the same one. That is what makes this an oracle test rather than a demonstration.

**How ADMM decouples the problem.** LASSO is hard because a least-squares term and an L1 term share a variable and want different things. ADMM *duplicates* the variable — $\min f(x) + g(z)$ subject to $x = z$ — so each term gets its own copy, and then enforces agreement with a dual. Read the three lines:

- `x = Q @ (A.T @ b + rho*(z - u))` — the smooth piece alone: a linear solve, with the factorisation `Q` **cached outside the loop**, so every iteration after the first is cheap.
- `z = soft(x + u, lam_reg/rho)` — the nonsmooth piece alone: Session 2's prox, reused verbatim.
- `u = u + x - z` — the **disagreement ledger**. It accumulates how far apart the two copies have been, and the longer they disagree the harder the dual pushes them together.

Neither subproblem is difficult. The difficulty was in their *combination*, and splitting removed it.

**Be honest about the comparison, though.** ADMM ran 300 iterations and FISTA ran 4000 — FISTA is the high-accuracy reference here, not the loser. ADMM's characteristic behaviour is to reach modest accuracy quickly and then converge slowly, so for a problem that fits in memory and needs many digits, FISTA is the better tool. Do not read this cell as ADMM winning.

**What ADMM wins is distribution.** The $x$-update decomposes across data blocks living on different machines: each solves its own piece locally, and only the consensus variable $z$ and the dual $u$ are exchanged. **One vector per round, not the data.** That is consensus ADMM, and it is the mathematical ancestor of the all-reduce pattern in [Distributed Training](../../Intro_GPU/Distributed_Training_2.ipynb) — the reason the template survives at datacenter scale where a monolithic solver cannot run at all.

**One wart worth knowing.** $\rho$ has to be chosen, and convergence *speed* is quite sensitive to it while the converged *answer* is not. Adaptive schemes that adjust $\rho$ from the primal and dual residuals are standard in practice. It is the one fiddly parameter in an otherwise clean method.

## 5. Conclusion

Duality turns optimality into a checkable certificate; prox maps extend gradient descent to the nonsmooth world (soft-thresholding, derived and brute-force-verified); ADMM splits problems into prox-sized pieces that negotiate via duals. The [Compressed Sensing](../../Intro_DSP/Compressed_Sensing.ipynb) notebook now has its complete theory.

---
## Where next

- [Manifold Optimization](./Manifold_Optimization.ipynb) — when the constraint is a surface, not a halfspace.
- [Distributed Training II](../../Intro_GPU/Distributed_Training_2.ipynb) — consensus at datacenter scale.